In [1]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import evaluate
import torch
import gc
import pandas as pd
import shutil
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [3]:
dataset_hf = load_dataset("JavaneseHonorifics/Unggah-Ungguh", "translation")
df_hf = dataset_hf["train"].to_pandas()[['javanese sentence', 'label']]

df_local_raw = pd.read_json("/kaggle/input/datasets/raditya0/javanese-synthesis/Javanese-Synthesis.json")

df_local_a = df_local_raw[['a_utterance', 'a_label']].rename(columns={'a_utterance': 'javanese sentence', 'a_label': 'label'})
df_local_b = df_local_raw[['b_utterance', 'b_label']].rename(columns={'b_utterance': 'javanese sentence', 'b_label': 'label'})
df_local = pd.concat([df_local_a, df_local_b], ignore_index=True).dropna()
df_local['label'] = df_local['label'].astype(int)

ds_hf = Dataset.from_pandas(df_hf)
ds_local = Dataset.from_pandas(df_local)

dataset_gabungan = concatenate_datasets([ds_hf, ds_local])

df_gabungan = dataset_gabungan.to_pandas()
print("Distribusi label dataset gabungan:")
print(df_gabungan["label"].value_counts(dropna=False))

dataset_trans = dataset_gabungan.train_test_split(test_size=0.2, seed=42)

dataset_conv = load_dataset("JavaneseHonorifics/Unggah-Ungguh", "conversation")
df_conv = dataset_conv["train"].to_pandas()

df_a = df_conv[['a utterance', 'a utterance category']].rename(columns={'a utterance': 'javanese sentence', 'a utterance category': 'label'})
df_b = df_conv[['b utterance', 'b utterance category']].rename(columns={'b utterance': 'javanese sentence', 'b utterance category': 'label'})
df_test_conv = pd.concat([df_a, df_b], ignore_index=True).dropna()
df_test_conv['label'] = df_test_conv['label'].astype(int)

conv_test_dataset = Dataset.from_pandas(df_test_conv)

daftar_model = [
    "xlm-roberta-base",
    "w11wo/javanese-bert-small",
    "indobenchmark/indobert-base-p1",
    "bert-base-multilingual-cased",
    "bert-base-uncased"
]

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

hasil_evaluasi = []

for model_name in daftar_model:
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(examples):
        return tokenizer(examples["javanese sentence"], padding="max_length", truncation=True, max_length=128)

    tokenized_trans = dataset_trans.map(tokenize_function, batched=True)
    if "label" in tokenized_trans["train"].column_names:
        tokenized_trans = tokenized_trans.rename_column("label", "labels")
    tokenized_trans.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    tokenized_conv = conv_test_dataset.map(tokenize_function, batched=True)
    if "label" in tokenized_conv.column_names:
        tokenized_conv = tokenized_conv.rename_column("label", "labels")
    tokenized_conv.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

    nama_folder_output = f"./klasifikasi_unggah_ungguh_{model_name.replace('/', '_')}"

    training_args = TrainingArguments(
        output_dir=nama_folder_output,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        save_total_limit=1,
        metric_for_best_model="accuracy",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_trans["train"],
        eval_dataset=tokenized_trans["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    
    eval_trans = trainer.evaluate()
    eval_conv = trainer.evaluate(eval_dataset=tokenized_conv)
    
    hasil_evaluasi.append({
        "Model": model_name,
        "Akurasi Test (Translation)": eval_trans.get("eval_accuracy"),
        "Akurasi Uji (Conversation)": eval_conv.get("eval_accuracy")
    })

    del model
    del trainer
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    shutil.rmtree(nama_folder_output, ignore_errors=True)

df_hasil = pd.DataFrame(hasil_evaluasi)
df_hasil.to_csv("rekap_evaluasi_model_lengkap.csv", index=False)
print(df_hasil)

model_terbaik_nama = df_hasil.loc[df_hasil["Akurasi Test (Translation)"].idxmax(), "Model"]
print(f"\nModel terbaik: {model_terbaik_nama}\nMelatih ulang model terbaik...")

tokenizer_best = AutoTokenizer.from_pretrained(model_terbaik_nama)

def tokenize_function_best(examples):
    return tokenizer_best(examples["javanese sentence"], padding="max_length", truncation=True, max_length=128)

tokenized_trans_best = dataset_trans.map(tokenize_function_best, batched=True)
if "label" in tokenized_trans_best["train"].column_names:
    tokenized_trans_best = tokenized_trans_best.rename_column("label", "labels")
tokenized_trans_best.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

model_best = AutoModelForSequenceClassification.from_pretrained(model_terbaik_nama, num_labels=4)

training_args_best = TrainingArguments(
    output_dir="./checkpoint_model_terbaik",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    save_total_limit=1,
    metric_for_best_model="accuracy",
    report_to="none"
)

trainer_best = Trainer(
    model=model_best,
    args=training_args_best,
    train_dataset=tokenized_trans_best["train"],
    eval_dataset=tokenized_trans_best["test"],
    compute_metrics=compute_metrics,
)

trainer_best.train()

trainer_best.save_model("./model_terbaik")
tokenizer_best.save_pretrained("./model_terbaik")
print("\nModel terbaik berhasil disimpan di folder: ./model_terbaik")
print(f"\nModel terbaik berhasil adalah {model_terbaik_nama}")

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2009 [00:00<?, ? examples/s]

Distribusi label dataset gabungan:
label
3    936
1    898
0    857
2    798
Name: count, dtype: int64


train-00000-of-00001.csv: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model terbaik berhasil disimpan di folder: ./model_terbaik

Model terbaik berhasil adalah w11wo/javanese-bert-small
